# Day 3: Forms, tools and a score

**Every record is well-formed. What would convince you the records are right?**

The short answer the day works towards is: not their shape.

Complete the [course setup](../../README.md#set-up-once) and select the **.venv** kernel. Exercises 3.2 and 3.3 send notes to the model through `model.py`; before them, check model access as the [course README](../../README.md#when-an-exercise-needs-ai-access) says. Use no other AI tool, and start no coding agent today. All notes and scenarios are fictional.

Use the cells below or the [terminal instructions](README.md). Startup keeps existing work. Each exercise ends when its check prints `Done:`; an initial `Not yet:` identifies work still to do. The reasons for each table's answers are in its **Why** toggle.

[Full reading and sources](https://catalyte-ai-irs-bpa-to6-weeks.vercel.app/week-01/#day-3)

### Today: from text to something a program can use.

**Day 2**

- A language model produces its answer one piece at a time
- That is why it can be fluent and wrong
- The system prompt, revised when it failed
- The model and its reasoning level, chosen with a reason

**How today runs**

- A reply checked against a schema
- A reply that names a function for your code to run
- A set of cases with a score the prompt is held to
- That score judges day 2's revisions and model choice

Section 2.2 wrote the extractor's system prompt and revised it when it failed; section 2.3 chose the model and its reasoning level and wrote down why.

## 3.1 Ask for structured output

*If the model's reply is guaranteed to match your schema, is it correct?*

Each Section plan shows the whole day; walk it once here.

### A schema guarantees shape, not content; it can cost accuracy.

**What it buys**

- A reply your code can rely on
- No required key left out
- No invalid enum value (enum: a field's fixed list of values)
- It limits which tokens may come at each step

**What it does not**

- OpenAI: “Structured Outputs can still contain mistakes.”
- Input unrelated to the schema can push it to hallucinate
- No source shows it makes a model invent less

**What it can cost**

- A 2024 study: reasoning declined under format restrictions
- Part of it: GPT 3.5 Turbo put the “answer” key before “reason”
- So measure the format on the evaluation set

The sources behind this chapter support something narrower. One vendor, OpenAI, says a schema means you need not worry about the model omitting a required key or hallucinating an invalid enum value; enum is the JSON Schema keyword for a field with a fixed list of values. One study found format restrictions helped on some classification tasks. Neither shows that a schema makes a model less likely to fill in a value the input does not contain. The paper behind Outlines, an open-source library for constraining model output, speaks of guaranteeing the structure of the generated text: structure, not content. OpenAI's guide says outright that Structured Outputs can still contain mistakes, and that the model will always try to adhere to the schema, which can result in hallucinations if the input is completely unrelated to it. What structured output does buy is a reply your code can rely on: Anthropic's page describes valid, parseable output for downstream processing.

A 2024 study of five models found a significant decline in reasoning abilities under format restrictions, and traced part of it to the order of keys: 100% of GPT 3.5 Turbo's JSON-mode responses placed the answer key before the reason key.

### Every field is required, and only the amount may be null.

*Null is what lets a note with no amount come back without an invented one.*

| Field | Type | May be null |
| --- | --- | --- |
| tax_year | integer | No |
| form | string | No |
| issue | string | No |
| amount_in_question | integer or null | Yes: a note may name none |
| next_action | string | No |

This is the case-notes extractor's schema, written in JSON Schema, the standard way to write what a valid reply looks like. Only amount_in_question may be null, because the lab's notes always state the other four fields. A field that can be absent in your own data needs null too.

The extractor's contract is three things: required, a type that allows null, and no extra keys. With required, an object is valid only if every name in the array is a property in it. For a value that may be absent, keep the field required and allow null; OpenAI's guide says it is possible to emulate an optional parameter by using a union type with null.

*Drawn: one field of the schema, with and without null; both keep it required.*

<img src="figures/null-lets-no-amount-pass.png" width="864" alt="Two columns for the field amount_in_question. Integer only: required; for a note that names an amount, the amount; for a note that names no amount, one valid move, put a number there, so the invented amount is the only answer that passes. Integer or null, marked in orange: required; for a note that names an amount, the amount; for a note that names no amount, null passes, so the reply can say there is none.">

*Allowing null in amount_in_question is what lets a note with no amount come back without an invented one.*

The figure draws the extractor's schema for one field, amount_in_question, with and without null. The null matters more than it looks. If amount_in_question were required and could only be an integer, a note with no amount would leave the model one valid move: put a number there.

*Source: the chapter, section 3.1, the extractor's schema; OpenAI, Structured Outputs guide*

### Validate every reply in your own code anyway.

*The guarantee has stated exceptions.*

- May fail the schema: a refusal, a cut-off, capitalization
- Tax year 2203 passes the schema, not your own check

<img src="figures/the-schema-then-your-own-checks.png" width="864" alt="A reply comes in from the left to a blue box, The schema; an arrow labelled passes leads on to an orange box, Your own check. From each box an arrow labelled fails drops to one band along the bottom: Counted as a failure, never repaired by guessing. Beside the schema&#x27;s arrow, in blue, what it stops: a refusal; cut off at the token limit; an enum value with different capitalization. Beside your own check&#x27;s arrow, in orange, what it stops: a tax year of 2203 is a valid integer.">

Anthropic's page says structured outputs guarantee schema compliance in most cases, and names three scenarios where the output may not match your schema: a refusal, a reply cut off at the token limit, and an enum value returned with different capitalization. And a reply can pass a schema and still be unusable; the JSON Schema specification says structural validation alone may be insufficient for an application to use certain values correctly. A tax year of 2203 is a valid integer, so only the extractor's own check, the orange gate, stops it. So the extractor validates every reply against the schema and against simple checks of its own, and a reply that fails is counted as a failure, never repaired by guessing.

*Source: Anthropic, Structured outputs; the JSON Schema specification; the chapter, section 3.1*

**Check:** In the 2024 study, “answer” came first. What did that cost?

- **A.** Nothing: the reply still matched the format
- **B.** It answered directly instead of reasoning first
- **C.** It left out required keys

<details><summary>Answer</summary>

**B.** With “answer” first, it answered before it reasoned: format can cost accuracy, so measure it.

</details>

**Discuss:** Guaranteed shape, guaranteed right?

- If the reply is guaranteed to match your schema, is it correct?
- Why may amount_in_question be null, and what if it could not be?
- If the schema is guaranteed, why validate the reply in your own code?

---

### 3.1 Check every reply

Make `check_reply` count bad replies as failures, while good replies and the lab's tests still pass.

Five of the eight replies in `replies/` failed `check_reply` in `extractor.py`. A teammate then changed the schema and `check_reply`, and now only two fail.

**Workspace:** `week-01/day-03/validate-replies/lab`. Paths below start here.

**Done when:** cell 3 prints `Done:`.

**1. Open the files.** Run cell 1. It opens `schema/output.schema.json` and `extractor.py`, the two files that the teammate changed. Each reply in `replies/` is named for its note in `notes/`.

In [ ]:
# 1. Start the exercise. Open the two files.
%cd -q "{_dh[0]}/validate-replies"
!test -d lab/.git || python3 ../../../reset.py validate-replies
%cd -q lab
!code schema/output.schema.json extractor.py

**2. Fix both files by hand.** Run cell 2. Expect `Failures: 2 of 8 replies.` Fewer failures do not mean better replies.

Use section 3.1 to explain each change.

| Problem | Change to make |
|---|---|
| The schema lets `next_action` be null, though the lab's notes always state a next action. Reply 0138 passes with null. | In `schema/output.schema.json`, change `"next_action": {"type": ["string", "null"]}` to `"next_action": {"type": "string"}`. |
| `check_reply` guesses a number from an amount written as text. | In `extractor.py`, remove the six lines from `# The model sometimes`. |
| Reply 0130 gives tax year 2203, after this year. The schema passes any integer. | In `extractor.py`, put `import datetime` above `import json`. Put the two lines below just above `return fields`, after the schema check. |

```python
    if fields["tax_year"] > datetime.date.today().year:
        raise ExtractionFailed("the tax year is after this year")
```

<details><summary>Why each change</summary>

| Problem | Why |
|---|---|
| The schema lets `next_action` be null, though the lab's notes always state a next action. Reply 0138 passes with null. | Note 0138 ends "Wait for the call.", but reply 0138 gives null. Every note states its next action, so null there is wrong. Section 3.1, below the table of fields: "Only amount_in_question may be null, because the lab's notes always state the other four fields." |
| `check_reply` guesses a number from an amount written as text. | These lines turn "40,000 of 95,000" in reply 0105 into 4000095000, a wrong amount. A guess can be wrong, so the reply must fail instead. Section 3.1, below the second figure: "a reply that fails is counted as a failure, never repaired by guessing" |
| Reply 0130 gives tax year 2203, after this year. The schema passes any integer. | Reply 0130 gives tax year 2203, but note 0130 names 2023. The schema passes any integer, so only your own check can fail a year that has not begun. After the schema check, `tax_year` is an integer, so `>` can compare it with this year. Section 3.1, in the list above the second figure: "Tax year 2203 passes the schema, not your own check" |

</details>

Save both files and run cell 2 again. Expect `Failures: 6 of 8 replies.`

In [ ]:
# 2. Count the failures.
%cd -q "{_dh[0]}/validate-replies/lab"
!python3 count_failures.py

**3. Check.** Run cell 3. If it prints `Not yet:`, address the findings and check again. The check reads your saved files, so no commit is needed.

In [ ]:
# 3. Check.
%cd -q "{_dh[0]}/validate-replies/lab"
!python3 ../check.py

**Reflect**

- The comment you removed names "18,400": should `check_reply` turn it into 18400, and why?
- Which wrong tax year does your own check still pass, and what would stop it?

[Worked answer](answers/validate-replies/) · [Terminal instructions](validate-replies/README.md)

To discard this exercise's edits and commits, run `!python3 "{_dh[0]}/../../reset.py" validate-replies` in a new cell.

---

## 3.2 Describe a function

*When a model calls a function, who runs the function?*

### Your code runs the function; the model only asks.

*OpenAI: “When the model calls a function, you must execute it and return the result.”*

<img src="figures/who-runs-the-function.png" width="864" alt="Three steps, left to right. You describe the functions: a name, a description, a schema for the arguments. The model, in grey, replies with a call: it only asks. Your code runs it, in orange, and decides what each call may do. A dashed line returns from your code to the model: your code returns the result.">

The round trip, with the model in grey and your code in orange: you describe the functions, each with a name, a description and a schema for its arguments; the model replies with a call; your code runs it and sends the result back. The call is a request that does nothing until your code acts on it.

*Source: OpenAI, Function calling guide; the chapter, section 3.2*

### That makes the function list a boundary.

*The model can ask only for what you defined.*

- Your code decides what each call may do
- A value the model must not choose stays in your code

<img src="figures/a-value-the-model-must-not-choose.png" width="864" alt="Two sides: The model, in grey, at the left; Your code at the right. Between them a blue dashed line, the function list, with one definition sitting on it in each row. Top row: submit_refund(order_id). A grey arrow, a call, runs into it from the model&#x27;s side carrying order_id, in ink: the model fills it. Bottom row: submit_refund(). A grey arrow, a call, runs into it from the model&#x27;s side carrying nothing; an orange arrow brings order_id into it from your code&#x27;s side: your code passes it.">

The same guide gives a related habit for arguments your code already knows: define submit_refund() with no parameters and pass the order_id in your code. The guide's reason is to offload work from the model. The same move keeps a value the model must not choose out of its hands, which is what the figure's two rows show: in the top row the model fills order_id, in the bottom row your code passes it.

*Source: OpenAI, Function calling guide; the chapter, section 3.2*

### A lookup is harmless; an action on a case belongs to a person.

*IRS policy: “Do not use GenAI to make binding determinations or actions on taxpayer rights without proper oversight.”*

| A function the model may call | What it does | Offer it? |
| --- | --- | --- |
| Look up what a form number means | Returns the form's title; records nothing | Yes: harmless |
| Record an action on a case | Writes an action into the case file | No: that call belongs to a person |

For the case-notes extractor, a function the model can call to look up what a form number means is harmless.

### Describe it for a colleague; then watch for invented arguments.

**Describe each function**

- Its purpose, and each parameter
- Could an intern use it from that alone?
- strict: true, so calls adhere to the schema
- OpenAI: “We recommend always enabling strict mode.”

**Invented arguments**

- A valid call can carry a guessed value
- Anthropic: Claude Sonnet “might also infer a reasonable value”
- The Berkeley Function Calling Leaderboard, a public benchmark, also tests irrelevance: tools available, none invoked
- Your set: notes where a call is right, and notes where none is

Your evaluation set should hold both: notes where a call is right, and notes where no call should be made.

**Check:** Which call does the Berkeley leaderboard count as correct?

- **A.** Any call whose JSON is valid
- **B.** The name matches exactly, and every argument is an allowed value
- **C.** Any call made whenever tools are available

<details><summary>Answer</summary>

**B.** A valid call can still carry a guessed value. Test notes where a call is right, and notes where none is.

</details>

**Discuss:** Who runs the function?

- When a model calls a function, who runs the function?
- Why does that make the list of functions a boundary?
- Which functions should the extractor never offer the model, and why?

---

### 3.2 Define one function

Offer the model only a harmless function, describe it, and test it where a call is right and where none is.

A teammate wrote the functions that the model can ask for when a caseworker asks about a case note. Fix the list before anyone uses it.

`ask.py` calls the model; use no other AI tool.

**Workspace:** `week-01/day-03/define-a-function/lab`. Paths below start here.

**Done when:** cell 3 prints `Done:`.

**1. Open the list.** Run cell 1. Expect the names `add_follow_up` and `lookup_form`; `functions.json` opens in the editor. The model sees this list, not the code in `tools.py` that runs each call.

In [ ]:
# 1. Start the exercise. Open the function list.
%cd -q "{_dh[0]}/define-a-function"
!test -d lab/.git || python3 ../../../reset.py define-a-function
%cd -q lab
!code functions.json
!grep '"name"' functions.json

**2. Fix the list by hand, then send it.** Use section 3.2 to explain each change. Copy each `\"` as it is: JSON needs a `\` before a quote inside a text.

| Problem | Change to make |
|---|---|
| `add_follow_up` writes an action into the case file; that call belongs to a person. | Remove its definition, from its `{` to its `},`. |
| The `description` of `lookup_form` is `TODO`. | In place of `TODO`: `Gives the title of a tax form, such as \"U.S. Individual Income Tax Return\" for 1040. Call it only when a question asks what a form is, and the note does not say it. It reads a table of titles. It writes nothing.` |
| The `description` of `form_number` is `TODO`. | In place of `TODO`: `The number of the form as the note writes it, without the word Form, such as \"5329\", \"1120-S\" or \"W-2\".` |

<details><summary>Why each change</summary>

| Problem | Why |
|---|---|
| `add_follow_up` writes an action into the case file; that call belongs to a person. | In `tools.py`, it writes the next action into the case file. IRS policy keeps such actions under a person's oversight. Section 3.2, the table of functions, row "Record an action on a case": "No: that call belongs to a person". |
| The `description` of `lookup_form` is `TODO`. | The model cannot see `tools.py`. So this text says what the function does, when to call it, and that it writes nothing. Notes 0106 and 0147 name a form. Only the question about 0106 asks what a form is. Section 3.2, list "Describe each function": "Could an intern use it from that alone?" |
| The `description` of `form_number` is `TODO`. | `tools.py` finds the title of `5329`, but nothing for `Form 5329`. So the description says what to send. Section 3.2, the same list: "Its purpose, and each parameter". |

</details>

Save `functions.json` and run cell 2: `ask.py` sends the two questions and saves the replies. Expect a call to `lookup_form` with `{"form_number": "5329"}` for note 0106, and no call for note 0147. If it prints `Not sent:`, fix what it names and run cell 2 again.

In [ ]:
# 2. Send the two questions with your function list.
%cd -q "{_dh[0]}/define-a-function/lab"
!python3 ask.py

**3. Check.** Run cell 3. If it prints `Not yet:`, address the findings, run cell 2 again, and check again. The check reads your saved list and its two replies, so no commit is needed.

In [ ]:
# 3. Check.
%cd -q "{_dh[0]}/define-a-function/lab"
!python3 ../check.py

**Reflect**

- When could `add_follow_up` go back on the list, and why?
- If a teammate adds `read_note(note_id)`, which reads any note, who should choose `note_id`, and why?
- Which third question would you add to test the list, and why?

[Worked answer](answers/define-a-function/) · [Terminal instructions](define-a-function/README.md)

To discard this exercise's edits and commits, run `!python3 "{_dh[0]}/../../reset.py" define-a-function` in a new cell. It removes the saved replies, so the next send calls the model twice.

---

## 3.3 Start the evaluation set

*Five notes right. How often is it right, and would you know next month?*

### Reading by eye fails in two ways; a labelled set fixes both.

*The answer that feels sufficient: I read the outputs and they looked fine.*

**Why reading by eye fails**

- Five is a handful, not a rate
- No memory: after next month's prompt edit, nothing tells you the five still pass
- Or that a sixth kind of note now fails

**The evaluation set**

- A fixed set of labelled cases, scored the same way every time
- From its first day, the likeliest to fail: no amount, two years, an ambiguous issue
- Easy notes that all pass measure nothing
- Small this week; it grows every week

Make the set hold failures, not just cases. Eugene Yan writes on evaluating language-model products: for pass or fail evaluations, most of the time, what matters is the fails, as these are the trust-busting defects.

### Start by looking, not by measuring.

*See outputs before you fully define criteria.*

- Each note beside its JSON and the JSON it should be
- A note on each wrong field; pass or fail, and why

<img src="figures/one-note-on-the-review-sheet.png" width="864" alt="A review sheet for Note 0212, a made-up lab note. Five fields in rows, each in blue, with two columns: produced and should be. tax_year, form, issue and next_action show the same grey value on both sides, joined by an equals sign. amount_in_question, on an orange band: produced 0, not equal to, should be null; beside it, in an orange box, the short note: invented: the note names none. At the foot, the label: pass, in grey, and fail, chosen, in white on black; why: the model put 0 in amount_in_question.">

Hamel Husain, writing from consulting work with more than thirty companies, calls error analysis the single most valuable activity in AI development. The order matters: you can't fully define your evaluation criteria until you've seen a wide range of outputs, but you need criteria to evaluate those outputs in the first place. His write-up reports client results without a published dataset, so take the method from it, not a benchmark. For the extractor, patterns appear quickly: amounts invented for notes that name none; a date in the follow-up line read as the tax year; an issue copied word for word rather than summarised.

Husain prefers a pass or fail label to a rating scale, whose middle points evaluators struggle to tell apart. A binary label does not lose the detail, because the detail has moved to the written critique that accompanies the judgment. Eugene Yan gives a starting format: a spreadsheet with columns for input, output, additional metadata that helps evaluate the output, and a new column for the label.

The chapter calls that a failure. Its cause, which the exercise's write-up asks for, is why it happened: the schema required the field and allowed only an integer, so the model had to put a number there.

*Source: Hamel Husain on error analysis; the chapter, section 3.3*

### Score each field, set the bar, and run it on every change.

**Score the fields, not the note**

- Precision: how many extracted values were correct
- Recall: how many expected values were found
- Per field, the score names the instruction to change

**Set the bar; run it on every change**

- Fields a person reads tolerate more misses
- Anthropic: an instruction passed weeks of evaluations; with other prompt changes, one broader evaluation then showed a 3% drop
- Its fix: evaluations per model, on every prompt change
- Open-source agent software: the prompt “tested in around 1% of cases”

Scoring each field separately is this course's choice for the lab, because it tells you which instruction in the prompt to change.

Yan: different use cases come with different levels of risk, so evaluation standards should be calibrated accordingly. A change that passes a narrow set can still be a regression. In its own incident report, Anthropic added a system prompt instruction to its coding tool to make answers shorter, after multiple weeks of internal testing and no regressions in the evaluations it ran. The report says the instruction did its damage in combination with other prompt changes, and does not say what the 3% is a percentage of. Its stated fix: run a broad suite of per-model evals, evaluations run for each model, for every system prompt change.

In open-source agent software, prompts are rarely tested: a study of unit tests sampled 377 test functions from 39 agent frameworks and 382 from 439 applications built on them, and found the prompt tested in around 1% of cases.

**Check:** A note names no amount; the reply invents one. Which drops?

- **A.** Recall: an expected value was not found
- **B.** Precision: an extracted value was not correct
- **C.** Neither: the reply still matches the schema

<details><summary>Answer</summary>

**B.** An invented amount on a note that names none lowers precision; a missed follow-up action lowers recall.

</details>

*The two measures separate the two ways an extractor can be wrong.*

<img src="figures/precision-and-recall.png" width="864" alt="Two columns. Precision, marked in orange: what proportion of extracted attributes were correct; in the lab&#x27;s scorer, how many extracted values were correct; lowered by an invented amount on a note that names none. Recall: what proportion of ground truth attributes were extracted; in the lab&#x27;s scorer, how many expected values were found; lowered by a missed follow-up action.">

*An invented amount lowers precision, the share of extracted values that were correct.*

The figure's questions are quoted from Eugene Yan; the lab's wording and the examples are this course's.

*Source: Eugene Yan, Task-Specific LLM Evals that Do & Don't Work, 2024; the chapter, section 3.3*

**Discuss:** How often is it right?

- What would you need before you could say how often it is right?
- How would you know next month that a change had made it worse?
- What is the difference between a failure and its cause?

---

### 3.3 Score the prompt

Name one failure of a teammate's prompt and its cause, revise the prompt once, and score it again on the same labelled set.

A teammate revised the extractor's prompt after reading five replies that looked right. Nobody has read the prompt's first score on the 20 labelled notes.

`evaluation/score.py` calls the model; use no other AI tool.

**Workspace:** `week-01/day-03/score-the-prompt/extractor`. Paths below start here.

**Done when:** cell 4 prints `Done:`.

**1. Open the files.** Run cell 1. It opens these files, which are also in VS Code's Explorer:

- `prompts/system.txt`: the teammate's prompt
- `evaluation/runs/1/score.txt`: its first score, with `form` at 0.00 for precision and recall
- `evaluation/failure.md`: your write-up

In [ ]:
# 1. Start the exercise. Open the files.
%cd -q "{_dh[0]}/score-the-prompt"
!test -d extractor/.git || python3 ../../../reset.py score-the-prompt
%cd -q extractor
!code prompts/system.txt evaluation/runs/1/score.txt evaluation/failure.md

**2. Write up the failure and revise the prompt by hand.** Use section 3.3 and day 2's section 2.2 to explain each answer.

| Part | What to write |
|---|---|
| The failure: one wrong value of the first score, after `Note:` and `Field:` in `evaluation/failure.md`. | `Note: 0147` and `Field: form` |
| Its cause: the lines of `prompts/system.txt` that led to it, after `Cause:`. | `Cause: 11, 17` |
| The change: the model writes `form` as the examples do. | In both example replies of `prompts/system.txt`, write `"form": "1040"` in place of `"form": "Form 1040"`. |

<details><summary>Why each answer</summary>

| Part | Why |
|---|---|
| The failure: one wrong value of the first score, after `Note:` and `Field:` in `evaluation/failure.md`. | In `evaluation/runs/1/score.txt`, all 20 wrong values are in `form`. Each label gives the right value: the form number only, such as "1040". Each reply adds "Form". So any note shows the failure, such as 0147. Section 3.3, the list "Score the fields, not the note": "Per field, the score names the instruction to change" |
| Its cause: the lines of `prompts/system.txt` that led to it, after `Cause:`. | Lines 11 and 17 of `prompts/system.txt` are the replies of the two examples. Both write `"form": "Form 1040"`. No line says how to write a form, so the model copies the examples. Day 2 showed the same with amounts. Day 2, section 2.2, the list "Examples teach the shape": "Every example with an amount teaches that notes have amounts" |
| The change: the model writes `form` as the examples do. | The model copies the examples. With "1040" in them, it writes the form number only, as the labels do. Day 2, section 2.2, under "Revise by editing the prompt, then re-score the whole set.": "A loop: run the set, score it, edit the prompt" |

</details>

Save both files and run cell 2. For a new prompt, it sends the 20 notes to the model. Expect `form` above 0.00 in the line that ends with `(this prompt)`; if not, cell 4's check gives the next change.

In [ ]:
# 2. Score your prompt.
%cd -q "{_dh[0]}/score-the-prompt/extractor"
!python3 evaluation/score.py

**3. Commit.** Set `commit = True` and run cell 3.

In [ ]:
# 3. Commit, when commit is True.
%cd -q "{_dh[0]}/score-the-prompt/extractor"
commit = False
if commit:
    !cd "{_dh[0]}/score-the-prompt/extractor" && git add prompts/system.txt evaluation && git commit -qm "Revise the prompt once"
else:
    print("After step 2, set commit = True. Then run this cell again.")

**4. Check.** Run cell 4. If it prints `Not yet:`, address the findings, commit with cell 3, and check again. The check scores your saved replies itself and calls no model.

In [ ]:
# 4. Check.
%cd -q "{_dh[0]}/score-the-prompt/extractor"
!python3 ../check.py

**Reflect**

- Why can a labelled set find what the teammate's five replies did not?
- When would a new line about the form be a better change than the examples?
- Which differences in your second score come from your change, and which from the run?

[Worked answer](answers/score-the-prompt/) · [Terminal instructions](score-the-prompt/README.md)

To discard this exercise's edits and commits, run `!python3 "{_dh[0]}/../../reset.py" score-the-prompt` in a new cell. It removes your prompt's saved replies, so the next score calls the model 20 times.

---

## Close

*Key points, and what day 4 builds*

### The reply is checked, the call is bounded, the prompt is scored.

- A schema guarantees shape, not content: validate every reply in your own code
- Allow null: without it, the invented amount is the only answer that passes
- Your code runs the function; the model only asks, so the list is a boundary
- The prompt is scored per field on a set that holds failures, on every change
- Write up a failure with its cause: the cause points at its own fix

### The prompt has a recorded score.

- Day 4: Git and the coding agent
- The repository, its remote copy and the diff
- The coding agents that write changes into them

Close the day. Day 4 turns to the repository, its remote copy and the diff, and the coding agents that write changes into them.